In [ ]:
from pathlib import Path
import json
import torch
import easyocr
import matplotlib.pyplot as plt
from PIL import Image

import warnings

warnings.filterwarnings("ignore", message=".*pin_memory.*")

# Combined train + test paths
TRAIN_IMG_DIR = Path("../data/raw/SROIE2019/train/img")
TRAIN_ENTITIES_DIR = Path("../data/raw/SROIE2019/train/entities")
TEST_IMG_DIR = Path("../data/raw/SROIE2019/test/img")
TEST_ENTITIES_DIR = Path("../data/raw/SROIE2019/test/entities")

reader = easyocr.Reader(["en"], gpu=torch.backends.mps.is_available())
print(f"MPS: {torch.backends.mps.is_available()}")

In [ ]:
def run_ocr(image_path: Path, confidence_threshold: float = 0.5) -> dict:
    detections = reader.readtext(str(image_path))
    filtered = [
        (bbox, text, conf)
        for bbox, text, conf in detections
        if conf >= confidence_threshold
    ]
    return {
        "raw_text": "\n".join([text for _, text, _ in filtered]),
        "detections": detections,
        "confidences": [conf for _, _, conf in detections],
    }

In [ ]:
sample_img = next(TRAIN_IMG_DIR.glob("X00016469612.jpg"))
ocr_output = run_ocr(sample_img)

In [ ]:
def visualize_ocr(image_path: Path, ocr_output: dict, figsize=(12, 14)):
    img = Image.open(image_path)
    fig, axes = plt.subplots(1, 2, figsize=figsize)

    axes[0].imshow(img)
    axes[0].set_title("Original")
    axes[0].axis("off")

    axes[1].imshow(img)
    axes[1].set_title("EasyOCR Detections")
    axes[1].axis("off")

    for bbox, text, conf in ocr_output["detections"]:
        xs = [p[0] for p in bbox] + [bbox[0][0]]
        ys = [p[1] for p in bbox] + [bbox[0][1]]
        color = "green" if conf >= 0.5 else "red"
        axes[1].plot(xs, ys, "-", color=color, linewidth=1)
        axes[1].text(
            bbox[0][0],
            bbox[0][1] - 5,
            f"{text} ({conf:.2f})",
            color="blue",
            fontsize=6,
            bbox=dict(facecolor="white", alpha=0.4, pad=1),
        )

    plt.tight_layout()
    plt.show()


visualize_ocr(sample_img, ocr_output)

In [ ]:
def load_gt(entities_dir: Path, image_path: Path) -> dict:
    gt_path = entities_dir / image_path.with_suffix(".txt").name
    return json.loads(gt_path.read_text())

In [ ]:
all_pairs = [
    (img, TRAIN_ENTITIES_DIR / img.with_suffix(".txt").name)
    for img in sorted(TRAIN_IMG_DIR.glob("*.jpg"))
] + [
    (img, TEST_ENTITIES_DIR / img.with_suffix(".txt").name)
    for img in sorted(TEST_IMG_DIR.glob("*.jpg"))
]

print(f"Total receipts: {len(all_pairs)}")

In [ ]:
gt = load_gt(TRAIN_ENTITIES_DIR, sample_img)

print(f"FILE: {sample_img.name}")
print(f"OCR:\n  " + ocr_output["raw_text"].replace("\n", "\n  "))
print("GT:")
for k, v in gt.items():
    print(f"  {k}: {v}")

In [ ]:
def aggregate_results(all_results):
    fields = ["company", "date", "address", "total"]
    n = len(all_results)
    summary = {}

    for field in fields:
        summary[field] = {
            "extraction_rate": sum(r[field]["extracted"] for r in all_results) / n,
            "avg_score": sum(r[field]["score"] for r in all_results) / n,
        }

    summary["overall"] = {
        "extraction_rate": sum(summary[f]["extraction_rate"] for f in fields)
        / len(fields),
        "avg_score": sum(summary[f]["avg_score"] for f in fields) / len(fields),
    }

    return summary

In [ ]:
import os
import base64
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.messages import HumanMessage

load_dotenv()

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [ ]:
def encode_image(image_path: Path) -> str:
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

In [ ]:
def improved_ocr(image_path: Path) -> str:
    encoded = encode_image(image_path)
    message = HumanMessage(
        content=[
            {"type": "image_url", "image_url": f"data:image/jpeg;base64,{encoded}"},
            {
                "type": "text",
                "text": "Extract all text you see in this receipt image exactly as it appears. Preserve the original content and structure.",
            },
        ]
    )
    return model.invoke([message]).content


# Test on sample
result = improved_ocr(sample_img)
print(f"FILE: {sample_img.name}")
print(f"OCR:\n  " + result.replace("\n", "\n  "))
print("GT:")
for k, v in gt.items():
    print(f"  {k}: {v}")

In [ ]:
def correct_text(raw_text: str) -> str:
    prompt = f"""You are correcting OCR text extracted from a scanned receipt.
Apply the following corrections while preserving all original content:

1. Rectify malformed words caused by OCR misreads:
   - Character confusion (e.g. '0' misread as 'O', '1' as 'l', 'rn' as 'm')
   - Broken words (e.g. 'To tal' should be 'Total')
   - Garbled text (e.g. 'sollor' should be 'seller')

2. Complete compound entities with missing components:
   - Use world knowledge to complete partial business names
     (e.g. 'MYDIN MO' should be completed to 'MYDIN MOHAMED HOLDINGS SDN BHD')
   - Complete partial addresses using surrounding context clues
     (e.g. a postcode or city name can help infer the full address)

3. Apply any other necessary corrections:
   - Normalize spacing and punctuation where clearly wrong
   - Fix split lines that belong together
   - Correct currency symbols or amount formatting if malformed

Important:
- Preserve all original content that does not need correction
- Do not add information that cannot be inferred from the text or world knowledge
- Return only the corrected text, nothing else

OCR text to correct:
{raw_text}"""

    return model.invoke([HumanMessage(content=prompt)]).content

In [ ]:
corrected = correct_text(ocr_output["raw_text"])

print(f"FILE:      {sample_img.name}")
print(f"RAW:\n  " + ocr_output["raw_text"].replace("\n", "\n  "))
print(f"CORRECTED:\n  " + corrected.replace("\n", "\n  "))
print("GT:")
for k, v in gt.items():
    print(f"  {k}: {v}")

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional


class ReceiptEntities(BaseModel):
    company: Optional[str] = Field(
        None, description="Full company or vendor name as it appears on the receipt"
    )
    date: Optional[str] = Field(
        None, description="Transaction date as it appears on the receipt"
    )
    address: Optional[str] = Field(
        None, description="Full address as a single string as it appears on the receipt"
    )
    total: Optional[str] = Field(
        None, description="Final total amount as it appears on the receipt"
    )

In [ ]:
entity_llm = model.with_structured_output(ReceiptEntities)


def extract_entities(text: str) -> ReceiptEntities:
    message = HumanMessage(
        content=f"Extract the entities from this receipt text:\n\n{text}"
    )
    return entity_llm.invoke([message])


# Test Tier 3 — entity LLM from EasyOCR text
easy_text = run_ocr(sample_img)["raw_text"]
entities = extract_entities(easy_text)
gt = load_gt(TRAIN_ENTITIES_DIR, sample_img)

print(f"FILE: {sample_img.name}")
print("ENTITIES:")
for k, v in entities.model_dump().items():
    print(f"  {k}: {v}")
print("GT:")
for k, v in gt.items():
    print(f"  {k}: {v}")

In [ ]:
def unified_pipeline(image_path: Path) -> ReceiptEntities:
    encoded = encode_image(image_path)
    prompt = """You are processing a scanned receipt image.
Extract the required entities directly from the image and in doing so:

1. Rectify malformed words caused by OCR misreads:
   - Character confusion (e.g. '0' misread as 'O', '1' as 'l', 'rn' as 'm')
   - Broken words (e.g. 'To tal' should be 'Total')
   - Garbled text (e.g. 'sollor' should be 'seller')

2. Complete compound entities with missing components:
   - Use world knowledge to complete partial business names
     (e.g. 'MYDIN MO' should be completed to 'MYDIN MOHAMED HOLDINGS SDN BHD')
   - Complete partial addresses using surrounding context clues
     (e.g. a postcode or city name can help infer the full address)

3. Apply any other necessary corrections:
   - Normalize spacing and punctuation where clearly wrong
   - Fix split lines that belong together
   - Correct currency symbols or amount formatting if malformed

Important:
- Do not add information that cannot be inferred from the image or world knowledge
- Extract only what is present or reasonably inferable from the receipt"""

    message = HumanMessage(
        content=[
            {
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{encoded}"},
            },
            {"type": "text", "text": prompt},
        ]
    )
    return entity_llm.invoke([message])


# Test unified pipeline
entities = unified_pipeline(sample_img)
gt = load_gt(TRAIN_ENTITIES_DIR, sample_img)

print(f"FILE: {sample_img.name}")
print("ENTITIES:")
for k, v in entities.model_dump().items():
    print(f"  {k}: {v}")
print("GT:")
for k, v in gt.items():
    print(f"  {k}: {v}")

In [ ]:
from rapidfuzz import fuzz

def evaluate_text_tier(raw_text, gt):
    results = {}
    for field, gt_value in gt.items():
        score = fuzz.partial_ratio(gt_value.lower(), raw_text.lower())
        results[field] = {"extracted": score >= 30, "score": score}
    return results


def evaluate_entity_tier(entities, gt):
    results = {}
    for field, gt_value in gt.items():
        pred = entities.get(field)
        results[field] = {
            "extracted": pred is not None,
            "score": (
                fuzz.token_sort_ratio(gt_value.lower(), pred.lower()) if pred else 0
            ),
        }
    return results

In [ ]:
easy_text = ocr_output["raw_text"]
gemini_text = improved_ocr(sample_img)

tier1 = evaluate_text_tier(easy_text, gt)
tier2 = evaluate_text_tier(gemini_text, gt)
tier3 = evaluate_entity_tier(extract_entities(correct_text(easy_text)).model_dump(), gt)
tier4 = evaluate_entity_tier(
    extract_entities(correct_text(gemini_text)).model_dump(), gt
)
tier5 = evaluate_entity_tier(unified_pipeline(sample_img).model_dump(), gt)

In [ ]:
tiers = [tier1, tier2, tier3, tier4, tier5]
labels = [
    "T1 (EasyOCR)",
    "T2 (Gemini OCR)",
    "T3 (EasyOCR+LLM)",
    "T4 (Gemini+LLM)",
    "T5 (Unified)",
]
fields = ["company", "date", "address", "total"]

col_w = 20
print(f"FILE: {sample_img.name}\n")
print(f"  {'field':10}" + "".join(f"{l:>{col_w}}" for l in labels))
print(f"  {'-'*10}" + "-" * (col_w * len(labels)))
for field in fields:
    scores = [
        f"{'✓' if t[field]['extracted'] else '✗'} {t[field]['score']:.1f}"
        for t in tiers
    ]
    print(f"  {field:10}" + "".join(f"{s:>{col_w}}" for s in scores))